# LCEL 방식 — 간단한 RAG

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 벡터스토어 및 Retriever 설정
vector_store = Chroma(
    collection_name = "docs",
    embedding_function=OpenAIEmbeddings(
        model="nomic-embed-text-v2", # LM Studio에 로드한 임베딩 모델 이름
        base_url="http://localhost:1234/v1", # LM Studio 로컬 서버 주소
        api_key="lm-studio", # 아무 값이나 가능 (LM Studio는 키 검증 안 함)
        check_embedding_ctx_length=False # 로컬 모델 호환성 위해 추가 권장
    ),
    persist_directory="./chroma_db"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_template(
    "컨텍스트:\n{context}\n\n질문: {question}\n\n"
    "위 컨텍스트를 바탕으로 답변하세요."
)

# LCEL 체인 구성
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

llm = ChatOpenAI(
    model="gemma-4-12b-it",      # LM Studio에 로드한 채팅 모델 ID
    base_url="http://localhost:1234/v1",  # LM Studio 로컬 서버 주소
    api_key="lm-studio",          # 아무 값이나 가능 (검증 안 함)
    temperature=0.7               # 필요에 따라 조정
)

# 2. 체인을 구성합니다. 딕셔너리 내부 요소는 쉼표(,)로 구분합니다.
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("LangGraph란 무엇인가요?")


# StateGraph 방식 — 유연한 RAG

In [ ]:
from typing import TypedDict, List
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    query: str
    retrieved_docs: List[Document]
    answer:str

def retrieve(state: RAGState) -> dict:
    """벡터스토어에서 관련 문서를 검색합니다."""
    vectorstore = Chroma(
        collection_name="docs",
        embedding_function=OpenAIEmbeddings(
            model="nomic-embed-text-v2", # LM Studio에 로드한 임베딩 모델 이름
            base_url="http://localhost:1234/v1", # LM Studio 로컬 서버 주소
            api_key="lm-studio", # 아무 값이나 가능 (LM Studio는 키 검증 안 함)
            check_embedding_ctx_length=False # 로컬 모델 호환성 위해 추가 권장
        ),
        persist_directory="./chroma_db",
    )
    docs = vectorstore.similarity_search(state["query"], k=3)
    return {"retrieve_docs": docs}

def generate(state: RAGState) -> dict:
    """검색된 문서를 바탕으로 답변을 생성합니다."""
    llm = ChatOpenAI(
        model="gemma-4-12b-it",      # LM Studio에 로드한 채팅 모델 ID
        base_url="http://localhost:1234/v1",  # LM Studio 로컬 서버 주소
        api_key="lm-studio",          # 아무 값이나 가능 (검증 안 함)
        temperature=0.7               # 필요에 따라 조정
    )
    context = "\n\n".join(d.page_content for d in state["retrieved_docs"])
    response = llm.invoke(
        f"컨텍스트:\n{context}\n\n질문: {state['query']}\n\n"
        "위 컨텍스트를 바탕으로 답변하세요."
    )
    return {"answer": response.content}

# StateGraph 구성
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()

# 실행
result = app.invoke({"query": "LangGraph란 무엇인가요?"})
print(result["answer"])

# PDF 문서 로딩

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드(페이지 단위로 분할)
loader = PyPDFLoader("./data/ADHD-KR_이민호.pdf")
documents = loader.load()

# 결과 확인
print(f"로드된 페이지 수: {len(documents)}")
print(f"첫 페이지 내용 미리보기: {documents[0].page_content[:200]}")
print(f"메타데이터: {documents[0].metadata}")

# 텍스트 파일 로딩

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/readme.txt", encoding="utf-8")
documents = loader.load()
print(f"문서 수: {len(documents)}")

# 웹 페이지 로딩

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://wikidocs.net/293312")
documents = loader.load()
print(f"로드된 문서 수: {len(documents)}")
print(f"내용 미리보기: {documents[0].page_content[:200]}")

# 문서 청킹 - RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # 청크 최대 길이(문자 수)
    chunk_overlap=200, # 인접 청크 간 겹침(문맥 유지)
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 문서 분할
chunks = splitter.split_documents(documents)

print(f"원본 문서 수: {len(documents)}")
print(f"생성된 청크 수: {len(chunks)}")
print(f"첫 번째 청크 길이: {len(chunks[0].page_content)}자")

# 전체 예제: PDF → 청크

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서 로드
loader = PyPDFLoader("./data/DB_fis-내가 작성한 이력서.pdf")
documents = loader.load()

# 2. 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)
chunks = splitter.split_documents(documents)

# 3. 결과 확인
print(f"원본 페이지: {len(documents)}개 → 청크: {len(chunks)}개")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- 청크 {i} (길이: {len(chunk.page_content)}자) ---")
    print(chunk.page_content[:100] + "...")
    print(f"메타데이터: {chunk.metadata}")

C:\Users\lmh\AppData\Local\Temp\ipykernel_17748\313728330.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


원본 페이지: 4개 → 청크: 9개

--- 청크 0 (길이: 667자) ---
이 민 호 남 만  24 세  24  0727 000994
 DB Inc  & DB FIS  2024 년   채 용연 계 형   인 턴 사 원   모 집
국적대한 민 국 생년월 일...
메타데이터: {'producer': 'Skia/PDF m127', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36 Edg/127.0.0.0', 'creationdate': '2024-08-18T12:20:21+00:00', 'title': '내가 작성한 이력서', 'moddate': '2024-08-18T12:20:21+00:00', 'source': './data/DB_fis-내가 작성한 이력서.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

--- 청크 1 (길이: 445자) ---
  대
학 력 사 항   추 가
 논 문   첨 부
경 력 사 항
 
포트 폴 리 오   첨 부
직 장경 력 재직   회 사   수 1개
티 앤 아 이 텍 2024 05 02   ...
메타데이터: {'producer': 'Skia/PDF m127', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36 Edg/127.0.0.0', 'creationdate': '2024-08-18T12:20:21+00:00', 'title': '내가 작성한 이력서', 'moddate': '2024-08-18T12:20:21+00:00', 'source': './data/DB_fis-내가 작성한 이력서.pdf', 'total_pages': 4, 'p

# OpenAI 임베딩 모델

In [ ]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1", # LM Studio 서버 주소
    api_key="lm-studio", # 아무 문자열이나 가능 (필수 인자라서 채움)
    model="nomic-embed-text", # LM Studio에서 로드한 모델 이름
    check_embedding_ctx_length=False, # 로컬 모델 호환성 위해 권장
)

# 텍스트를 벡터로 변환
vector = embeddings.embed_query("LangGraph란 무엇인가요?")
print(f'벡터 차원:{len(vector)}')
print(f'벡터 샘플:{vector[:5]}')

# Chroma 벡터 데이터베이스

In [4]:
from langchain_chroma import Chroma, vectorstores
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서로드 및 청킹
loader = PyPDFLoader("./data/DB_fis-내가 작성한 이력서.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(documents)

# 2. 벡터스토어 생성
vectorstores = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(
        base_url="http://localhost:1234/v1",
        model="text-embedding-bge-m3",
        api_key="lm-studio",
        check_embedding_ctx_length=False
    ),
    collection_name="resume_docs",
    persist_directory="./chroma_db"
)

print(f"저장된 문서 수: {vectorstores._collection.count()}")

저장된 문서 수: 18


# 기존 벡터스토어 로드

In [5]:
from langchain_chroma import Chroma, vectorstores
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma(
    collection_name="resume_docs",
    embedding_function=OpenAIEmbeddings( # 기존 벡터스토어를 로드할 때 embedding_function 파라미터를 반드시 지정해야 함
        base_url="http://localhost:1234/v1",
        model="text-embedding-bge-m3",
        api_key="lm-studio",
        check_embedding_ctx_length=False
    ),
    persist_directory="./chroma_db"
)

print(f"로드된 문서 수: {vectorstore._collection.count()}")

로드된 문서 수: 18


# 유사도 검색

## 기본 유사도 검색

In [ ]:
results = vectorstores.similarity_search("대학교", k=3)

for i, doc in enumerate(results):
    print(f"\n--- 결과 {i+1} ---")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")

## 유사도 점수와 함께 검색

In [ ]:
results_with_scores = vectorstores.similarity_search_with_score(
    "대학교", k=3
)

for doc, score in results_with_scores:
    print(f"점수: {score:.4f} | {doc.page_content[:80]}...")

# Retriever 패턴

In [ ]:
# 기본 Retriever
retriever = vectorstores.as_retriever(search_kwargs={"k": 5})

# Retriever로 검색
docs = retriever.invoke("이 이력서를 쓴 사람의 이름")
print(f"검색된 문서 수: {len(docs)}")

# MMR (Maximal Marginal Relevance) — 다양성 확보
mmr_retriever = vectorstores.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10},
)
docs = mmr_retriever.invoke("이 이력서를 쓴 사람의 이름")
print(f"검색된 문서 수(MMR): {len(docs)}")

# 검색 최적화

## 쿼리 확장

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

def expand_query(query: str, n: int = 3) -> list[str]:
    """LLM으로 쿼리를 확장합니다."""
    llm = init_chat_model(
        model="gemma-4-12b-it",           # LM Studio에 로드한 모델 ID
        model_provider="openai",          # OpenAI 호환 API 사용
        base_url="http://localhost:1234/v1", # LM Studio 로컬 서버 주소
        api_key="lm-studio",              # 필수 입력값 (임의의 문자열 가능)
        temperature=0.7                   # 추가 하이퍼파라미터 설정
    )
    prompt = ChatPromptTemplate.from_template(
        "다음 질문에 대해 의미는 같지만 표현이 다른 검색 쿼리를 "
        "{n}개 생성하세요. 각 쿼리를 줄바꿈으로 구분하세요.\n\n"
        "원본 질문: {query}"
    )
    response = llm.invoke(prompt.format(query=query, n=n))
    expanded = response.content.strip().split("\n")
    return [q.strip() for q in expanded if q.strip()][:n]

# 사용 예시
queries = expand_query("LangGraph에서 상태 관리는 어떻게 하나요?")
for q in queries:
    print(f"- {q}")
# - LangGraph의 State 설계 방법
# - StateGraph에서 TypedDict로 상태를 정의하는 방법
# - LangGraph 노드 간 데이터 공유 방식


- LangGraph 상태 관리 방법
- LangGraph state management guide
- LangGraph에서 State를 어떻게 유지하나요?


## 하이브리드 검색 - Dense 검색(임베딩 유사도) + Sparse 검색(키워드 매칭)

## EnsembleRetriever 구현

In [6]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Dense Retriever (벡터 검색)
vectorstore = Chroma(
    collection_name="docs",
    embedding_function=OpenAIEmbeddings(
        base_url="http://localhost:1234/v1",
        model="text-embedding-bge-m3",
        api_key="lm-studio",
        check_embedding_ctx_length=False
    ),
    persist_directory="./chroma_db",
)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Sparse Retriever (BM25 키워드 검색)
# documents: 이전에 청킹한 Document 객체 리스트
sparse_retriever = BM25Retriever.from_documents(documents, k=5)

# 앙상블 Retriever (가중치: Dense 70%, Sparse 30%)
ensemble = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

# 검색 실행
results = ensemble.invoke("출신 대학교")
print(f"검색된 문서 수: {len(results)}")
for doc in results[:3]:
    print(f"- {doc.page_content[:100]}...")

검색된 문서 수: 4
- 2   지 원 직 무 와   관 련 하 여   자신의   경 쟁 력 을   나타 낼   수   있는   경 험 이 나   이 력   등 을   기 술 하 십 시 오     학 교 ...
-  
컴 퓨 터 활 용 능력
 
수상경 력
교 육 이 수 사 항
그 렙
리 눅 스   시스 템   및   커 널   전문 가   과정
이 수 기 간 2023 10 02   2024 ...
-   대
학 력 사 항   추 가
 논 문   첨 부
경 력 사 항
 
포트 폴 리 오   첨 부
직 장경 력 재직   회 사   수 1개
티 앤 아 이 텍 2024 05 02   ...


## 메타데이터 필터링

In [11]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma(
    collection_name="docs",
    embedding_function=OpenAIEmbeddings(
        base_url="http://localhost:1234/v1",
        model="text-embedding-bge-m3",
        api_key="lm-studio",
        check_embedding_ctx_length=False
    ),
    persist_directory="./chroma_db",
)

# 특정 소스 파일에서만 검색
results = vectorstore.similarity_search(
    "상태 관리",
    k=3,
    filter={"source": "./data/DB_fis-내가 작성한 이력서.pdf"},
)
print(results)

# 특정 페이지 범위에서 검색
results = vectorstore.similarity_search(
    "상태 관리",
    k=3
)
print(results)

[]
[]


# 검색 결과 평가

## 간단한 검색 품질 평가

In [13]:
def evaluate_retrieval(retriever, test_cases: list[dict]) -> dict:
    """검색 품질을 간단히 평가합니다.

    Args:
        test_cases: [{"query": "질문", "expected_keywords": ["핵심어1", "핵심어2"]}]
    """
    total = len(test_cases)
    hits = 0

    for case in test_cases:
        docs = retriever.invoke(case["query"])
        combined = " ".join(d.page_content for d in docs)

        # 기대 키워드가 검색 결과에 포함되는지 확인
        found = any(kw in combined for kw in case["expected_keywords"])
        if found:
            hits += 1

    recall = hits / total if total > 0 else 0
    return {"recall": recall, "hits": hits, "total": total}

# 사용 예시
test_cases = [
    {"query": "출신 대학교", "expected_keywords": ["대학교", "학교"]},
    {"query": "노드 함수 작성법", "expected_keywords": ["add_node", "노드"]},
]

result = evaluate_retrieval(ensemble, test_cases)
print(f"Recall: {result['recall']:.2%} ({result['hits']}/{result['total']})")


Recall: 0.00% (0/2)
